In [ ]:
!pip install "pandas<3.0.0"

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# Import common libraries
import numpy as np
import pandas as pd

# Import MNE processing
import mne
from mne.preprocessing.nirs import optical_density, beer_lambert_law
from mne.preprocessing.nirs import optical_density, beer_lambert_law, scalp_coupling_index, temporal_derivative_distribution_repair


# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.statistics import statsmodels_to_results
from mne_nirs.channels import get_short_channels, get_long_channels
from mne_nirs.channels import picks_pair_to_idx
from mne_nirs.visualisation import plot_glm_group_topo
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.visualisation import plot_glm_surface_projection
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals

# Import StatsModels
import statsmodels.formula.api as smf

# Import Plotting Library
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from itertools import compress
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from scipy import signal

from scipy.stats.mstats import winsorize
import statsmodels
import statsmodels.stats.multitest as smt


# Import other randoms
import h5py
from scipy import interpolate

import ipyevents
import pyvistaqt
import ipympl

In [ ]:
# create lowpass filter
from scipy.signal import butter, filtfilt
def lowpass_filter(data, cutoff, fs, order=4):
    nyquist = 0.5 * fs  # Nyquist frequency
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data
    

In [ ]:
# add time lag
import numpy as np
from scipy import signal

# Find optimal lag between two signals based on cross-correlation
def find_optimal_lag(nirs_data, aux_data, max_lag_samples=30):
    nirs = nirs_data - np.mean(nirs_data)
    aux = aux_data - np.mean(aux_data)
    correlation = np.correlate(aux, nirs, mode='full')
    lags = signal.correlation_lags(len(aux), len(nirs), mode='full')

    # Limit lags to a specific window
    valid_idx = (lags >= 0) & (lags <= max_lag_samples)
    correlation = correlation[valid_idx]
    lags = lags[valid_idx]

    best_lag = lags[np.argmax(np.abs(correlation))]
    best_corr = correlation[np.argmax(np.abs(correlation))]

    return best_lag, best_corr

# Create a lagged version of a dataset
def create_lagged_matrix(data, lags):
    return np.hstack([np.roll(data, shift=lag, axis=0) for lag in lags])


In [ ]:
# Define the individual analysis function to be used for each individual dataset
def individual_analysis(bids_path, ID):

    #raw_intensity = read_raw_bids(bids_path=bids_path, verbose=False)
    raw_intensity = mne.io.read_raw_snirf(fname=bids_path, verbose=False, optode_frame="mri")

    # delete unnecessary annotations
    raw_intensity.annotations.delete(raw_intensity.annotations.description == '15')
    raw_intensity.annotations.delete(raw_intensity.annotations.description == '1')
    raw_intensity.annotations.delete(raw_intensity.annotations.description == '2')
    raw_intensity.annotations.delete(raw_intensity.annotations.description == '23')
    raw_intensity.annotations.delete(raw_intensity.annotations.description == '27')



    
    raw_intensity.annotations.rename({'3': 'Congruent', # congruent listening
                                         '13': 'CongruentPreQ', # congruent speech processing
                                         '7': 'Incongruent', # incongruent listening
                                         '17': 'IncongruentPreQ'}) # incongruent speech processing

    
    # sanitize event names
    raw_intensity.annotations.description[:] = [
        d.replace('/', '_') for d in raw_intensity.annotations.description]

    # Convert signal to haemoglobin and resample

    raw_od = optical_density(raw_intensity)
    sci = scalp_coupling_index(raw_od, h_freq=1.35, h_trans_bandwidth=0.1)
    raw_od.info['bads'] = list(compress(raw_od.ch_names, sci < 0.7)) #change back to 0.7
    raw_od.interpolate_bads()
    raw_haemo = beer_lambert_law(raw_od, ppf=0.1)

    raw_haemo.resample(3.0)


    # IDing short chans
    sht_chans = get_short_channels(raw_haemo)
    raw_haemo = get_long_channels(raw_haemo)

    # Create a design matrix
    design_matrix = make_first_level_design_matrix(raw_haemo, 
                                                   drift_model='cosine',
                                                   high_pass=0.015,  # Must be specified per experiment
                                                   hrf_model='spm',
                                                   stim_dur=8.0)


    design_matrix["ShortHbO"] = np.mean(sht_chans.copy().pick(picks="hbo").get_data(), axis=0)
    design_matrix["ShortHbR"] = np.mean(sht_chans.copy().pick(picks="hbr").get_data(), axis=0)

    
    physio_cols = []


    with h5py.File(bids_path, 'r') as dat:
        for n in [13,14,15,16,17,18]: 
            try:
                aux = np.array(dat.get(f'nirs/aux{n}/dataTimeSeries'))
                aux_time = np.array(dat.get(f'nirs/aux{n}/time'))
    
                # Interpolate to match fNIRS timing
                aux_data_interp = interpolate.interp1d(
                    aux_time, aux, axis=0, bounds_error=False, fill_value='extrapolate'
                )
                aux_data_matched_to_fnirs = aux_data_interp(raw_haemo.times)
                name_raw = np.array(dat.get(f'nirs/aux{n}/name'))
                if name_raw.ndim == 0:
                    name = name_raw.item().decode()
                else:
                    name = name_raw[0].decode()
    
                # Add to design matrix
                design_matrix[name] = aux_data_matched_to_fnirs
                physio_cols.append(name)

            except Exception as e:
                print(f"Skipping aux{n} due to error: {e}")

    physio_data = design_matrix[physio_cols].values
    physio_data = StandardScaler().fit_transform(physio_data)

    fs = 1 / np.median(np.diff(raw_haemo.times))
    max_lag_seconds = 10
    max_lag_samples = int(max_lag_seconds * fs)
    lags = np.arange(-max_lag_samples, max_lag_samples + 1)

    # below is where we identify each "best lag" for each aux signal
    for i, col in enumerate(physio_cols):
        best_lag, _ = find_optimal_lag(physio_data[:, 0], physio_data[:, i], max_lag_samples=max_lag_samples)        
        physio_data[:, i] = np.roll(physio_data[:, i], best_lag)

    physio_lagged = create_lagged_matrix(physio_data, lags)
    valid_idx = max(abs(lags))
    physio_lagged = physio_lagged[valid_idx:-valid_idx]

    fnirs_data = raw_haemo.copy().pick(picks=["hbo", "hbr"]).get_data().T
    fnirs_data = StandardScaler().fit_transform(fnirs_data)
    fnirs_lagged = fnirs_data[valid_idx:-valid_idx]
    

    #CCA component
    cca = CCA(n_components=2, tol=0.3)
    physio_c, fnirs_c = cca.fit_transform(physio_lagged, fnirs_lagged)

    # Matching lengths of nirs and physio data; making sure raw_haemo contains both and is the same length
    # Only best lags used
    start = valid_idx
    end = -valid_idx if valid_idx > 0 else None
    design_matrix = design_matrix.iloc[start:end].reset_index(drop=True)
    raw_haemo.crop(tmin=raw_haemo.times[start], tmax=raw_haemo.times[end - 1] if end is not None else raw_haemo.times[-1])

    # Pair physiology data to design matrix
    design_matrix["CCA1"] = physio_c[:, 0]
    design_matrix["CCA2"] = physio_c[:, 1]

    glm_est = run_glm(raw_haemo, design_matrix)

    # ID channels to group together for ROIs 
    PFC = [[5, 5], [7, 5], [6, 4], [5, 4], [5, 2], [4, 2], [1, 2], [2, 2], [1, 9], [2, 10], [4, 9], [4, 10], [1, 1], [2, 3]]  # dorsolateral prefrontal cortex and prefrontal 
    MC = [[3, 6], [9, 8], [8, 7], [8, 6], [9, 6], [11, 11], [12, 12], [3, 7], [3, 8]] # supp motor and primary motor 
    IFG = [[13, 1], [6, 1], [7, 3], [14, 3], [15, 1], [16, 3], [10, 1]]
   



    groups = dict(PFC = picks_pair_to_idx(raw_haemo, PFC),
                  MC = picks_pair_to_idx(raw_haemo, MC),
                  IFG = picks_pair_to_idx(raw_haemo, IFG))


    # Extract channel metrics
    cha = glm_est.to_dataframe()
    roi = glm_est.to_dataframe_region_of_interest(groups, design_matrix.columns, demographic_info=True)
    
    # Define contrasts for conditions
    contrast_matrix = np.eye(design_matrix.shape[1])
    basic_conts = dict([(column, contrast_matrix[i])
                        for i, column in enumerate(design_matrix.columns)])

    #change the following two lines (comment or uncomment) to select which contrast to compute
    contrast_SpchSil = basic_conts['Incongruent'] - basic_conts['Congruent'] # listening contrast
    # contrast_SpchSil = basic_conts['IncongruentPreQ'] - basic_conts['CongruentPreQ'] # speech processing contrast

    
    contrast = glm_est.compute_contrast(contrast_SpchSil)
    con = contrast.to_dataframe()
    
    # Add the participant ID to the dataframes
    roi["ID"] = cha["ID"] = con["ID"] =  ID

    # Convert to uM for nicer plotting below.
    cha["theta"] = [t * 1.e6 for t in cha["theta"]]
    roi["theta"] = [t * 1.e6 for t in roi["theta"]]
    con["effect"] = [t * 1.e6 for t in con["effect"]]

    return raw_haemo, roi, cha, con, design_matrix

# analysis

In [ ]:
df_roi = pd.DataFrame()  # To store region of interest results
df_cha = pd.DataFrame()  # To store channel level results
df_con = pd.DataFrame()  # To store channel level contrast results


for sub in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,  14, 15, 16, 17, 18]:  # change range values to specify the number of subject recordings to be analyzed,,
    ID = 'C%03d' % sub
    bids_path = BIDSPath(subject="C%03d" % sub,
                         session="01", 
                         datatype="nirs",
                         root="C:\\Users\\abbie\\Box\\BRAiN Lab\\current projects\\openQs-project\\sourcedata",
                         extension= ".snirf")
    raw_haemo, roi, cha, con,  design_matrix = individual_analysis(bids_path, ID) 

    
    df_roi = pd.concat([df_roi, roi], ignore_index=True)
    df_cha = pd.concat([df_cha, cha], ignore_index=True)
    df_con = pd.concat([df_con, con], ignore_index=True)

In [ ]:
grp_results = df_roi.query("Condition in ['Congruent', 'Incongruent', 'CongruentPreQ', 'IncongruentPreQ']")
grp_results = grp_results.query("Chroma in ['hbo']")

sns.catplot(x="Condition", y="theta", col="ID", hue="ROI", data=grp_results, col_wrap=5, errorbar=None, palette="muted", height=4, s=10)

In [ ]:
# df_roi.to_csv("roi_grouplevel_data.csv") #uncomment to get a full dataframe of roi weighted averages for each condition, though this is uploaded on OSF

In [ ]:
#prepare and run the LMEM
grp_results = df_roi.query("Condition in ['Congruent', 'Incongruent', 'CongruentPreQ', 'IncongruentPreQ']")
# grp_results2 = grp_results.query("Chroma in ['hbo']") #can uncomment if you just want to look at oxygenated hemoglobin

roi_model = smf.mixedlm("theta ~ -1 + ROI:Condition:Chroma",
                        grp_results, groups=grp_results["ID"]).fit(method='nm')
roi_model.summary()

In [ ]:
params = roi_model.params
bse = roi_model.bse
z_values = params / bse
pvalues = roi_model.pvalues
conf_int = roi_model.conf_int()

# Combine into a single dataframe with your desired column names
results_df = pd.DataFrame({
    "Coef.": params,
    "Std.Err.": bse,
    "z": z_values,
    "P>|z|": pvalues,
    "[0.025": conf_int[0],
    "0.975]": conf_int[1]
})

# Save LMEM output to CSV
# results_df.to_csv("roi_grouplevel_LMEM.csv")

In [ ]:
# read in individual level ROI contrast file uploaded on OSF for ease of LMEM computing. 

df_con2 = pd.read_csv('Individual ID ROI-level HbO and HbR Contrasts.csv')# contrast dataframe

In [ ]:
df_con2 = df_con2.query("Chroma in ['hbo']") #can comment if you want to look at both hbo and hbr

# add "Chroma" below as a fixed effect to add hbo + hbr to the model if you want to look at both of them
con_model = smf.mixedlm("ContrastEffect_SpeechProcessing~ -1 + ROI", # change to "ContrastEffect_Listening" for the listening contrast LMEM
                        df_con2, groups=df_con2["ID"]).fit(method='nm')
con_model.summary()

In [ ]:
params2 = con_model.params
bse = con_model.bse
z_values = params2 / bse
pvalues = con_model.pvalues
conf_int = con_model.conf_int()

# Combine into a single DataFrame with desired column names
results_df = pd.DataFrame({
    "Coef.": params2,
    "Std.Err.": bse,
    "z": z_values,
    "P>|z|": pvalues,
    "[0.025": conf_int[0],
    "0.975]": conf_int[1]
})

# Save contrast LMEM output to CSV
# results_df.to_csv("contrast_speechprocessing.csv")

In [ ]:
# Group topographic visualization 

# change conditions to your own conditions and channels/ROIs to your own 


fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 10),
                         gridspec_kw=dict(width_ratios=[1, 1]))

groups_single_chroma = dict(
    PFC = picks_pair_to_idx (raw_haemo.copy().pick(picks='hbo'),[[5, 5], [7, 5], [6, 4], [5, 4], [5, 2], [4, 2], [1, 2], [2, 2], [1, 9], [2, 10], [4, 9], [4, 10], [1, 1], [2, 3]] , on_missing = 'warning'),
    MC = picks_pair_to_idx (raw_haemo.copy().pick(picks= 'hbo'),[[3, 6], [9, 8], [8, 7], [8, 6], [9, 6], [11, 11], [12, 12], [3, 7], [3, 8]], on_missing = 'warning'),
    IFG = picks_pair_to_idx (raw_haemo.copy().pick(picks = 'hbo'), [[13, 1], [6, 1], [7, 3], [14, 3], [15, 1], [16, 3], [10, 1]], on_missing = 'warning'))


ch_summary = df_cha.query("Condition in ['Incongruent', 'Congruent']")  #####Change these to your conditions####
ch_summary = ch_summary.query("Chroma in ['hbo']")

# Run group level model and convert to dataframe
ch_model = smf.mixedlm("theta ~ -1 + ch_name:Chroma:Condition",
                       ch_summary, groups=ch_summary["ID"]).fit(method='nm')
ch_model_df = statsmodels_to_results(ch_model)




# deoxygenated hemoglobin (HbR)
ch_summary = df_cha.query("Condition in ['Congruent', 'Incongruent']")         
ch_summary = ch_summary.query("Chroma in ['hbr']")

# Run group level model and convert to dataframe
ch_model = smf.mixedlm("theta ~ -1 + ch_name:Chroma:Condition",
                       ch_summary, groups=ch_summary["ID"]).fit(method='nm')
ch_model_df = statsmodels_to_results(ch_model)




# plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
con_summary = df_con.query("Chroma in ['hbo']") # contrast topographic plots

# Run group level model and convert to dataframe
con_model = smf.mixedlm("effect ~ -1 + ch_name:Chroma",
                        con_summary, groups=con_summary["ID"]).fit(method='nm')
con_model_df = statsmodels_to_results(con_model,
                                      order=raw_haemo.copy().pick(
                                          picks="hbo").ch_names)

#contrast plot for each ROI
plot_glm_group_topo(raw_haemo.copy().pick(picks="hbo").pick(groups_single_chroma['IFG']),
                    con_model_df, colorbar=True, vlim=(-5, 5), axes=axes)

plot_glm_group_topo(raw_haemo.copy().pick(picks="hbo").pick(groups_single_chroma['PFC']),
                    con_model_df, colorbar=True, vlim=(-5, 5), axes=axes)

plot_glm_group_topo(raw_haemo.copy().pick(picks="hbo").pick(groups_single_chroma['MC']),
                    con_model_df, colorbar=True, vlim=(-5, 5), axes=axes)

In [ ]:
#contrast cortical projection plots

# add channels to make the 3D projection match LMEM output. The LMEM uses ROI-level data which is composed of weighted averages of channels
PFC = ['S5_D5 hbo', 'S7_D5 hbo', 'S6_D4 hbo', 'S5_D4 hbo', 'S5_D2 hbo', 'S4_D2 hbo', 'S1_D2 hbo', 'S2_D2 hbo', 'S1_D9 hbo', 'S2_D10 hbo', 'S4_D9 hbo', 'S4_D10 hbo', 'S1_D1 hbo', 'S2_D3 hbo']
MC = ['S3_D6 hbo', 'S9_D8 hbo', 'S8_D7 hbo', 'S8_D6 hbo', 'S9_D6 hbo', 'S11_D11 hbo', 'S12_D12 hbo', 'S3_D7 hbo', 'S3_D8 hbo']
IFG = [ 'S13_D1 hbo', 'S6_D1 hbo', 'S7_D3 hbo', 'S14_D3 hbo', 'S15_D1 hbo', 'S16_D3 hbo', 'S10_D1 hbo']
picks = list(set(PFC + MC + IFG))

# this will create a plot for whichever contrast condition was included in the GLM. You will need to change between listening or the speech processing section to get your desired contrast plot
subjects_dir = mne.datasets.sample.data_path() / 'subjects'
clim = dict(kind='value', pos_lims=(0, 8, 11))
brain = plot_glm_surface_projection(raw_haemo.copy().pick("hbo"),
                                    con_model_df, clim=clim, view='lateral',
                                    colorbar=True, size=(800, 700), subjects_dir=subjects_dir,
                                   picks = picks)
brain.add_text(0.05, 0.95, "incongurent-congruent listening contrast", 'title', font_size=16, color='k') # change title based on the contrast you're using 


clim = dict(kind='value', pos_lims=(0, 11.5, 17))